# Quickstart — independent CdMC end-to-end

This notebook is a 60-second tour of FactorTail. We:

1. Build three heavy-tailed margins.
2. Run the §3 independent summed CdMC at a deep threshold.
3. Compare to the first-order and second-order asymptotics.
4. Read the Bernstein CI off the result.

For the manuscript-section map and a comprehensive overview see
[Concepts](../concepts.md). For the full estimator menu see
[Estimator families](../models.md).

## Build the marginals

We use three iid Pareto($\alpha=2$) margins. The DGP is the simplest
instance of *Family I* in §8.

In [ ]:
import numpy as np
from factortail.utils.tails import ParetoTail

margs = [ParetoTail(alpha=2.0, scale=1.0) for _ in range(3)]
[(m.alpha, m.scale, m.c) for m in margs]

## Run the CdMC estimator

At $x=20$ with $n=20\,000$ replicates.

In [ ]:
from factortail.cdmc import independent_cdmc

res = independent_cdmc(margs, x=20.0, n=20_000, seed=42)
print(f'mu_hat   = {res.mu_hat:.6e}')
print(f'std err  = {res.standard_error:.6e}')
print(f'rel SE   = {res.rel_sd:.4f}')
print(f'95% CI   = [{res.ci_low:.6e}, {res.ci_high:.6e}]')
print(f"envelope = {res.extra['envelope']:.6e}  (deterministic upper bound on Z)")

## Compare to the first-order and second-order asymptotics

`thm:sum-equivalence` says $P(S_N>x) \sim \sum_i \overline F_i(x)$.
`thm:second-order` adds the leave-one-out mean-shift correction. Both
are closed-form for Pareto.

In [ ]:
from factortail.utils.regular_variation import (
    first_order_sum_tail,
    second_order_sum_tail,
)

x = np.array([20.0])
fo = float(first_order_sum_tail(margs, x)[0])
so = float(second_order_sum_tail(margs, x)[0])
print(f'first_order  = {fo:.6e}')
print(f'second_order = {so:.6e}  (closer to mu_hat)')
print(f'CdMC mu_hat  = {res.mu_hat:.6e}')
print(f'ratio mu_hat/first_order  = {res.mu_hat/fo:.4f}')

The CdMC sits between the first-order line and the empirical reference, with
the second-order line tracking it closely — the textbook prediction
of the manuscript's §3.

## What's next

- **[Dependent designs](02_dependent_designs.ipynb)** — common-shock,
  copula, MRV.
- **[Real-data pipeline](03_real_data_pipeline.ipynb)** — rolling
  VaR/ES on the Fama–French panel.
- **[CLI reference](../cli.md)** — `factortail run`, `factortail
  validate-run`, `factortail repro`.
- **[Reproducibility](../reproducibility.md)** — seeds, hashing, App. G
  replacement contract.